# Notebook 01: WLASL-2000 Dataset EDA & Split Preparation

Notebook này thực hiện toàn bộ quá trình khám phá dữ liệu (EDA), phân tích phân bố classes, video, signers và trích xuất danh sách 5 classes khởi đầu theo official WLASL splits.

### Nội dung:
1. **Kaggle environment**: Khởi tạo môi trường, repo và thư mục outputs.
2. **Dataset discovery**: Tự động nhận diện thư mục dataset `WLASL-2000 Resized` trong `/kaggle/input`.
3. **Read WLASL metadata**: Đọc và phẳng hóa annotations từ `WLASL_v0.3.json`.
4. **Dataset statistics**: Thống kê tổng quan số từ vựng, số lượng mẫu video.
5. **Class distribution**: Phân tích phân bố tần suất các classes (từ vựng).
6. **Video statistics**: Kiểm tra tính sẵn sàng của video trên disk (tồn tại/thiếu, độ dài, FPS).
7. **Signer distribution**: Phân tích số lượng người ký (signer identity).
8. **Official train/val/test analysis**: Đánh giá các tập split chính thức và kiểm tra data leakage của signer.
9. **Select initial 5 classes**: Chọn 5 từ vựng có nhiều mẫu video nhất.
10. **Visual inspection**: Trực quan hóa một số frame mẫu từ các video đã chọn.
11. **Save selected split CSV**: Xuất các file `train.csv`, `val.csv`, `test.csv` và dừng lại.


---
## 1. Kaggle Environment & Paths Setup


In [ ]:
import os
import sys
from pathlib import Path

# Kiểm tra môi trường Kaggle
IN_KAGGLE = Path('/kaggle/working').exists()
print(f"Running in Kaggle: {IN_KAGGLE}")

# Repo setup (nếu chạy trên Kaggle)
REPO_URL = 'https://github.com/ChienPM-27/SignLanguageReconition_ML.git'
BRANCH = 'main'
REPO_DIR = Path('/kaggle/working/SignLanguageReconition_ML') if IN_KAGGLE else Path('.').resolve()

if IN_KAGGLE:
    if not REPO_DIR.exists():
        print(f"Cloning repo from {REPO_URL} (branch: {BRANCH})...")
        !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
    else:
        print(f"Pulling latest changes on {BRANCH}...")
        !cd {REPO_DIR} && git checkout {BRANCH} && git pull
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

# Thiết lập thư mục outputs (chỉ ghi vào /kaggle/working hoặc outputs/ local)
OUTPUT_ROOT = Path('/kaggle/working/asl_outputs') if IN_KAGGLE else Path('outputs').resolve()
SPLIT_OUTPUT_DIR = OUTPUT_ROOT / 'splits'
PLOTS_OUTPUT_DIR = OUTPUT_ROOT / 'plots'

for d in [OUTPUT_ROOT, SPLIT_OUTPUT_DIR, PLOTS_OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Outputs will be saved to: {OUTPUT_ROOT}")


---
## 2. Dataset Discovery in `/kaggle/input`


In [ ]:
kaggle_input = Path('/kaggle/input')

# Tìm kiếm thư mục chứa dataset WLASL
dataset_candidates = [d for d in kaggle_input.iterdir() if d.is_dir() and 'wlasl' in d.name.lower()] if kaggle_input.exists() else []

if dataset_candidates:
    DATA_ROOT = dataset_candidates[0]
    print(f"Detected WLASL Dataset Directory: {DATA_ROOT}")
else:
    # Fallback mặc định theo slug Kaggle hoặc local path
    DATA_ROOT = Path('/kaggle/input/wlasl2000-resized') if IN_KAGGLE else Path('data').resolve()
    print(f"Defaulting DATA_ROOT to: {DATA_ROOT}")

print("\n--- Files & Folders in DATA_ROOT ---")
if DATA_ROOT.exists():
    for p in DATA_ROOT.iterdir():
        print(f"  - {p.name} ({'Directory' if p.is_dir() else 'File'})")
else:
    print(f"[WARNING] DATA_ROOT '{DATA_ROOT}' not found! Hãy đảm bảo dataset 'sttaseen/wlasl2000-resized' đã được add vào Notebook.")


---
## 3. Read WLASL Metadata


In [ ]:
import json
import pandas as pd
import numpy as np

# Tìm file JSON metadata chính (thường là WLASL_v0.3.json hoặc nslt_2000.json)
json_files = list(DATA_ROOT.glob('*.json')) if DATA_ROOT.exists() else []
print(f"Found JSON candidates: {[f.name for f in json_files]}")

metadata_json_path = None
for f in json_files:
    if 'v0.3' in f.name.lower() or 'wlasl' in f.name.lower():
        metadata_json_path = f
        break
if not metadata_json_path and json_files:
    metadata_json_path = json_files[0]

print(f"Using Metadata File: {metadata_json_path}")

with open(metadata_json_path, 'r', encoding='utf-8') as f:
    raw_metadata = json.load(f)

# Phẳng hóa annotations thành DataFrame
records = []
if isinstance(raw_metadata, list):
    for entry in raw_metadata:
        gloss = entry.get('gloss')
        for inst in entry.get('instances', []):
            row = dict(inst)
            row['gloss'] = gloss
            records.append(row)
elif isinstance(raw_metadata, dict):
    for k, v in raw_metadata.items():
        if isinstance(v, dict):
            row = dict(v)
            row['video_id'] = k
            records.append(row)

df_all = pd.DataFrame(records)
print(f"Total metadata records loaded: {len(df_all):,}")
df_all.head(5)


---
## 4. Dataset Statistics


In [ ]:
total_instances = len(df_all)
total_glosses = df_all['gloss'].nunique()
instances_per_gloss = df_all['gloss'].value_counts()

print("=" * 50)
print("WLASL DATASET OVERVIEW STATISTICS")
print("=" * 50)
print(f"Total Video Instances : {total_instances:,}")
print(f"Total Unique Glosses  : {total_glosses:,}")
print(f"Mean samples / gloss  : {instances_per_gloss.mean():.2f}")
print(f"Median samples / gloss: {instances_per_gloss.median():.2f}")
print(f"Min samples / gloss   : {instances_per_gloss.min()}")
print(f"Max samples / gloss   : {instances_per_gloss.max()}")
print(f"Std samples / gloss   : {instances_per_gloss.std():.2f}")


---
## 5. Class Distribution


In [ ]:
import matplotlib.pyplot as plt

top_20 = instances_per_gloss.head(20)
bottom_10 = instances_per_gloss.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot Top-20 classes
axes[0].barh(top_20.index[::-1], top_20.values[::-1], color='steelblue')
axes[0].set_title('Top 20 Most Frequent Glosses')
axes[0].set_xlabel('Number of Video Instances')

# Histogram of instances per gloss
axes[1].hist(instances_per_gloss.values, bins=30, color='coral', edgecolor='black')
axes[1].set_title('Distribution of Samples per Gloss')
axes[1].set_xlabel('Samples Count')
axes[1].set_ylabel('Number of Classes')

plt.tight_layout()
plt.savefig(PLOTS_OUTPUT_DIR / 'class_distribution.png', dpi=150)
plt.show()


---
## 6. Video Statistics (Disk Availability, Resolution, FPS)


In [ ]:
import cv2

# Tìm thư mục chứa video mp4
video_dirs = list(DATA_ROOT.glob('**/videos')) + list(DATA_ROOT.glob('**/*video*'))
VIDEO_DIR = video_dirs[0] if video_dirs else DATA_ROOT
print(f"Videos Directory: {VIDEO_DIR}")

# Kiểm tra video tồn tại thực tế trên disk
sample_vids = df_all['video_id'].astype(str).tolist()
existing_on_disk = []

print("Verifying video existence on disk...")
for vid_id in sample_vids[:1000]:  # Kiểm tra nhanh 1000 mẫu đầu
    vid_file = VIDEO_DIR / f"{vid_id}.mp4"
    if vid_file.exists():
        existing_on_disk.append(vid_id)

avail_ratio = len(existing_on_disk) / 1000.0 * 100
print(f"Availability check (sample 1000): {len(existing_on_disk)}/1000 exists ({avail_ratio:.1f}%)")

# Kiểm tra FPS & Resolution mẫu từ 5 video đầu tiên tồn tại
print("\n--- Sample Video Properties ---")
for vid_id in existing_on_disk[:5]:
    cap = cv2.VideoCapture(str(VIDEO_DIR / f"{vid_id}.mp4"))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    print(f"Video {vid_id}: Resolution = {w}x{h}, FPS = {fps:.1f}, Frame Count = {frames}")


---
## 7. Signer Distribution


In [ ]:
if 'signer_id' in df_all.columns:
    unique_signers = df_all['signer_id'].dropna().nunique()
    signer_counts = df_all['signer_id'].value_counts()
    print(f"Total unique signers: {unique_signers}")
    print(f"Top 5 signers with most recordings:\n{signer_counts.head(5)}")
else:
    print("Column 'signer_id' is not present in metadata annotations.")


---
## 8. Official Train / Val / Test Analysis & Leakage Check


In [ ]:
print("Official Splits Distribution across full dataset:")
print(df_all['split'].value_counts(dropna=False))

if 'signer_id' in df_all.columns:
    train_signers = set(df_all[df_all['split'] == 'train']['signer_id'].dropna())
    val_signers = set(df_all[df_all['split'] == 'val']['signer_id'].dropna())
    test_signers = set(df_all[df_all['split'] == 'test']['signer_id'].dropna())

    print(f"\nUnique Signers in Train : {len(train_signers)}")
    print(f"Unique Signers in Val   : {len(val_signers)}")
    print(f"Unique Signers in Test  : {len(test_signers)}")
    print(f"Signer overlap Train <-> Val : {len(train_signers.intersection(val_signers))}")
    print(f"Signer overlap Train <-> Test: {len(train_signers.intersection(test_signers))}")
    print("NOTE: Official WLASL benchmark split aims to test generalization; observing signer overlap is essential context.")


---
## 9. Select Initial 5 Classes


In [ ]:
# Tìm các gloss có nhiều mẫu nhất và đồng thời có video tồn tại trên disk
available_files_set = set(f.stem for f in VIDEO_DIR.glob('*.mp4')) if VIDEO_DIR.exists() else set()
print(f"Total mp4 files found in VIDEO_DIR: {len(available_files_set):,}")

if available_files_set:
    df_available = df_all[df_all['video_id'].astype(str).isin(available_files_set)].copy()
else:
    df_available = df_all.copy()

# Chọn 5 classes có nhiều video nhất
NUM_CLASSES = 5
top_5_classes = df_available['gloss'].value_counts().head(NUM_CLASSES).index.tolist()
print(f"\nSelected Initial {NUM_CLASSES} Classes: {top_5_classes}")

df_selected = df_available[df_available['gloss'].isin(top_5_classes)].copy()

summary_table = pd.crosstab(df_selected['gloss'], df_selected['split'], margins=True)
print("\n--- Split Summary for Selected 5 Classes ---")
display(summary_table) if 'display' in dir() else print(summary_table)


---
## 10. Visual Inspection of Sample Frames


In [ ]:
fig, axes = plt.subplots(1, len(top_5_classes), figsize=(18, 4))

for i, cls in enumerate(top_5_classes):
    sample_vid_id = str(df_selected[df_selected['gloss'] == cls]['video_id'].iloc[0])
    vid_path = VIDEO_DIR / f"{sample_vid_id}.mp4"
    
    if vid_path.exists():
        cap = cv2.VideoCapture(str(vid_path))
        total_f = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        # Đọc frame ở giữa video
        cap.set(cv2.CAP_PROP_POS_FRAMES, max(0, total_f // 2))
        ret, frame = cap.read()
        cap.release()
        
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            axes[i].imshow(frame_rgb)
            axes[i].set_title(f"Gloss: '{cls}'\nID: {sample_vid_id}")
            axes[i].axis('off')
        else:
            axes[i].text(0.5, 0.5, f"Error reading {sample_vid_id}", ha='center')
    else:
        axes[i].text(0.5, 0.5, f"{cls}\n(File missing)", ha='center')

plt.tight_layout()
plt.savefig(PLOTS_OUTPUT_DIR / 'sample_frames_inspection.png', dpi=150)
plt.show()


---
## 11. Save Selected Split CSV & Finish EDA


In [ ]:
# Thêm đường dẫn file video vào DataFrame để tiện cho Notebook 02
df_selected['video_path'] = df_selected['video_id'].apply(lambda x: str(VIDEO_DIR / f"{x}.mp4"))

# Lưu các file CSV split chuẩn
for split_name in ['train', 'val', 'test']:
    split_df = df_selected[df_selected['split'] == split_name]
    out_file = SPLIT_OUTPUT_DIR / f"{split_name}.csv"
    split_df.to_csv(out_file, index=False)
    print(f"Saved {split_name} split ({len(split_df)} samples) to: {out_file}")

# Lưu file tổng hợp 5 classes
all_selected_file = SPLIT_OUTPUT_DIR / 'selected_5_classes.csv'
df_selected.to_csv(all_selected_file, index=False)
print(f"Saved complete selected 5 classes dataframe ({len(df_selected)} samples) to: {all_selected_file}")

# Nếu trong repo có thư mục data/splits, copy thêm vào đó để tiện lưu trữ
repo_split_dir = REPO_DIR / 'data' / 'splits'
if repo_split_dir.exists():
    for split_name in ['train', 'val', 'test']:
        df_selected[df_selected['split'] == split_name].to_csv(repo_split_dir / f"{split_name}.csv", index=False)
    print(f"Also mirrored splits to repository directory: {repo_split_dir}")

print("\nEDA & Dataset Preparation hoàn tất! Sẵn sàng cho Notebook 02 (Landmark Extraction).")
